# Incident Response Runbook: Persistence - Account Manipulation

**Tactic:** Persistence
**Technique:** Account Manipulation (T1098)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Account Manipulation activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime, timedelta
import sys
import os

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from misp.misp_integration import MISPIntegration
from iris.iris_integration import IRISIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Step 1: Initial Alert Analysis & Detection
print("=" * 60)
print("STEP 1: Initial Alert Analysis & Detection")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
misp = MISPIntegration()
iris = IRISIntegration()
shuffle = ShuffleIntegration()

# Create incident case in IRIS
incident_data = {
    'title': 'Account Manipulation for Persistence - T1098',
    'description': 'Potential account manipulation activities for persistence detected',
    'severity': 'HIGH',
    'category': 'Persistence',
    'tags': ['account-manipulation', 'persistence', 'privilege-escalation', 't1098']
}

case_id = iris.create_case(incident_data)
print(f" Created IRIS case: {case_id}")

# Detection queries for account manipulation indicators
print("\n Executing account manipulation detection queries...")

# Splunk query for account creation/modification events
account_creation_query = """
index=* source="ActiveDirectory" (EventCode=4720 OR EventCode=4722 OR EventCode=4724 OR EventCode=4728)
| stats count by host, user, target_user, EventCode
| where count > 2
| sort -count
"""

creation_results = splunk.execute_query(account_creation_query)
print(f" Found {len(creation_results)} account creation/modification events")

# Splunk query for PowerShell account manipulation
powershell_query = """
index=* "Set-ADUser" OR "Set-ADAccountPassword" OR "Add-ADGroupMember" OR "Set-ADAccountControl"
| where user != "system" AND user != "administrator"
| stats count by host, user, command
| where count > 1
| sort -count
"""

powershell_results = splunk.execute_query(powershell_query)
print(f" Found {len(powershell_results)} PowerShell account manipulation commands")

# Splunk query for group membership changes
group_change_query = """
index=* source="ActiveDirectory" (EventCode=4728 OR EventCode=4729 OR EventCode=4732 OR EventCode=4733 OR EventCode=4756 OR EventCode=4757)
| stats count by host, user, target_group, EventCode
| where count > 3
| sort -count
"""

group_results = splunk.execute_query(group_change_query)
print(f" Found {len(group_results)} group membership changes")

# Splunk query for account attribute modifications
attribute_query = """
index=* source="ActiveDirectory" (EventCode=5136 OR EventCode=5137)
| search "userAccountControl" OR "pwdLastSet" OR "accountExpires" OR "userPrincipalName"
| stats count by host, user, attribute, old_value, new_value
| where count > 2
| sort -count
"""

attribute_results = splunk.execute_query(attribute_query)
print(f" Found {len(attribute_results)} account attribute modifications")

# CrowdStrike query for suspicious account activities
cs_account_query = """
event_simpleName=UserAccountCreated OR event_simpleName=UserAccountModified OR event_simpleName=GroupModified
| where UserName != "system" AND UserName != "administrator"
| stats count by ComputerName, UserName, TargetUserName
| where count > 1
"""

cs_account_results = crowdstrike.execute_query(cs_account_query)
print(f" Found {len(cs_account_results)} suspicious account activities")

# Check for known account manipulation IOCs in MISP
account_indicators = misp.search_indicators("account-manipulation", limit=50)
print(f" Retrieved {len(account_indicators)} account manipulation indicators from MISP")

# Compile detection results
detection_summary = {
    'case_id': case_id,
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Persistence',
    'technique': 'Account Manipulation (T1098)',
    'severity': 'HIGH',
    'indicators': {
        'account_creations': len(creation_results),
        'powershell_commands': len(powershell_results),
        'group_changes': len(group_results),
        'attribute_modifications': len(attribute_results),
        'suspicious_activities': len(cs_account_results),
        'misp_indicators': len(account_indicators)
    },
    'affected_accounts': list(set([r.get('target_user', r.get('TargetUserName', 'unknown')) 
                                  for r in creation_results + cs_account_results])),
    'suspicious_users': list(set([r.get('user', r.get('UserName', 'unknown')) 
                                 for r in powershell_results + group_results + attribute_results + cs_account_results]))
}

print(f"\n📊 Detection Summary:")
print(f"  Case ID: {detection_summary['case_id']}")
print(f"  Severity: {detection_summary['severity']}")
print(f"  Affected Accounts: {len(detection_summary['affected_accounts'])}")
print(f"  Suspicious Users: {len(detection_summary['suspicious_users'])}")
print(f"  Account Creations: {detection_summary['indicators']['account_creations']}")
print(f"  PowerShell Commands: {detection_summary['indicators']['powershell_commands']}")
print(f"  Group Changes: {detection_summary['indicators']['group_changes']}")

# Trigger Shuffle workflow for automated response
if detection_summary['indicators']['account_creations'] > 0 or detection_summary['indicators']['powershell_commands'] > 0:
    shuffle.trigger_workflow('account_manipulation_response', detection_summary)
    print(" Triggered automated account manipulation response workflow")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

isolated_systems = []
disabled_accounts = []
revoked_sessions = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        isolation_result = crowdstrike.isolate_host(system['device_id'])
        if isolation_result.get('status') == 'success':
            isolated_systems.append(system['hostname'])
            print(f"   Isolated system: {system['hostname']}")
        else:
            print(f"   Failed to isolate {system['hostname']}: {isolation_result}")
    except Exception as e:
        print(f"   Isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Disable manipulated accounts
print(f"\n[ACTION] Disabling manipulated accounts...")
unique_manipulated_accounts = list(set([activity.get('target_user', '') for activity in suspicious_activities if activity.get('target_user')]))
for account in unique_manipulated_accounts:
    try:
        account_disable = trigger_workflow(
            workflow_name="disable_compromised_account",
            parameters={
                'username': account,
                'reason': 'Account manipulation detected',
                'disable_logins': True,
                'preserve_for_forensic': True
            }
        )
        if account_disable.get('status') == 'success':
            disabled_accounts.append(account)
            print(f"   Disabled account: {account}")
        else:
            print(f"   Failed to disable account {account}: {account_disable}")
    except Exception as e:
        print(f"   Account disabling failed for {account}: {e}")

# Revoke active sessions for manipulated accounts
print(f"\n[ACTION] Revoking active sessions for manipulated accounts...")
for account in unique_manipulated_accounts:
    try:
        session_revoke = trigger_workflow(
            workflow_name="revoke_account_sessions",
            parameters={
                'username': account,
                'reason': 'Account manipulation incident',
                'force_logout': True,
                'invalidate_tokens': True
            }
        )
        if session_revoke.get('status') == 'success':
            revoked_sessions.append(account)
            print(f"   Revoked sessions for: {account}")
        else:
            print(f"   Failed to revoke sessions for {account}: {session_revoke}")
    except Exception as e:
        print(f"   Session revocation failed for {account}: {e}")

# Enable enhanced account monitoring
print(f"\n[ACTION] Enabling enhanced account monitoring...")
try:
    account_monitoring = trigger_workflow(
        workflow_name="enhanced_account_monitoring",
        parameters={
            'duration': '72h',
            'alert_on_manipulation': True,
            'alert_on_privilege_changes': True,
            'target_accounts': unique_manipulated_accounts
        }
    )
    if account_monitoring.get('status') == 'success':
        print(f"   Enhanced account monitoring enabled for {len(unique_manipulated_accounts)} accounts")
    else:
        print(f"   Failed to enable account monitoring: {account_monitoring}")
except Exception as e:
    print(f"   Account monitoring setup failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_systems)} systems isolated")
print(f"  - {len(disabled_accounts)} accounts disabled")
print(f"  - Sessions revoked for {len(revoked_sessions)} accounts")
print(f"  - Enhanced account monitoring enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_backdoors = []
reset_accounts = []
restored_permissions = []

# Analyze account manipulation with threat intelligence
print(f"\n[ACTION] Analyzing account manipulation with threat intelligence...")
account_analysis = {}
for activity in suspicious_activities:
    account_name = activity.get('target_user', '') if activity.get('target_user') else 'unknown'
    if account_name not in account_analysis:
        try:
            misp_analysis = search_iocs(account_name, type='username')
            account_analysis[account_name] = {
                'threat_level': len(misp_analysis) if misp_analysis else 0,
                'indicators': misp_analysis[:3] if misp_analysis else []
            }
            print(f"   Analyzed account: {account_name} (threat level: {account_analysis[account_name]['threat_level']})")
        except Exception as e:
            print(f"   Account analysis failed for {account_name}: {e}")

# Remove backdoors and persistence mechanisms
print(f"\n[ACTION] Removing backdoors and persistence mechanisms...")
for system in affected_systems:
    try:
        # Scan for and remove account manipulation backdoors
        scan_result = crowdstrike.scan_and_remove(
            system['device_id'],
            signatures=['account_backdoors', 'persistence_mechanisms', 'privilege_escalation_tools']
        )
        if scan_result.get('removed_files', []):
            removed_backdoors.extend(scan_result['removed_files'])
            print(f"   Removed {len(scan_result['removed_files'])} backdoors from {system['hostname']}")
    except Exception as e:
        print(f"   Backdoor removal failed for {system.get('hostname', 'unknown')}: {e}")

# Reset compromised accounts
print(f"\n[ACTION] Resetting compromised accounts...")
for account in unique_manipulated_accounts:
    try:
        account_reset = trigger_workflow(
            workflow_name="reset_compromised_account",
            parameters={
                'username': account,
                'reset_password': True,
                'reset_permissions': True,
                'remove_from_groups': True,
                'reason': 'Account manipulation incident'
            }
        )
        if account_reset.get('status') == 'success':
            reset_accounts.append(account)
            print(f"   Reset account: {account}")
        else:
            print(f"   Failed to reset account {account}: {account_reset}")
    except Exception as e:
        print(f"   Account reset failed for {account}: {e}")

# Restore proper permissions and group memberships
print(f"\n[ACTION] Restoring proper permissions and group memberships...")
for account in unique_manipulated_accounts:
    try:
        permission_restore = trigger_workflow(
            workflow_name="restore_account_permissions",
            parameters={
                'username': account,
                'restore_baseline_permissions': True,
                'remove_unauthorized_groups': True,
                'validate_group_memberships': True
            }
        )
        if permission_restore.get('status') == 'success':
            restored_permissions.append(account)
            print(f"   Restored permissions for: {account}")
        else:
            print(f"   Failed to restore permissions for {account}: {permission_restore}")
    except Exception as e:
        print(f"   Permission restoration failed for {account}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        verify_result = crowdstrike.verify_security_posture(system['device_id'])
        verification_results.append({
            'system': system['hostname'],
            'status': 'clean' if verify_result.get('threats_detected', 0) == 0 else 'threats_remaining',
            'threats': verify_result.get('threats_detected', 0)
        })
        status = "" if verify_result.get('threats_detected', 0) == 0 else ""
        print(f"  {status} Verification for {system['hostname']}: {verify_result.get('threats_detected', 0)} threats detected")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_backdoors)} backdoors removed")
print(f"  - {len(reset_accounts)} accounts reset")
print(f"  - Permissions restored for {len(restored_permissions)} accounts")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
validated_accounts = []
restored_access = []

# Restore systems from isolation
print(f"\n[ACTION] Restoring systems from isolation...")
for system in isolated_systems:
    try:
        # Find system details
        system_info = next((s for s in affected_systems if s['hostname'] == system), None)
        if system_info:
            restore_result = crowdstrike.restore_host(system_info['device_id'])
            if restore_result.get('status') == 'success':
                restored_systems.append(system)
                print(f"   Restored system: {system}")
            else:
                print(f"   Failed to restore {system}: {restore_result}")
    except Exception as e:
        print(f"   System restoration failed for {system}: {e}")

# Validate account integrity
print(f"\n[ACTION] Validating account integrity...")
for account in unique_manipulated_accounts:
    try:
        validation_checks = [
            "Account exists and is active",
            "Proper group memberships",
            "Correct permission levels",
            "No unauthorized privileges"
        ]

        account_validation = trigger_workflow(
            workflow_name="validate_account_integrity",
            parameters={
                'username': account,
                'check_permissions': True,
                'check_group_memberships': True,
                'verify_baseline_compliance': True
            }
        )
        if account_validation.get('status') == 'valid':
            validated_accounts.append(account)
            print(f"   Account integrity validated: {account}")
        else:
            print(f"   Account integrity issues detected for {account}: {account_validation}")
    except Exception as e:
        print(f"   Account validation failed for {account}: {e}")

# Restore legitimate account access
print(f"\n[ACTION] Restoring legitimate account access...")
for account in unique_manipulated_accounts:
    try:
        access_restore = trigger_workflow(
            workflow_name="restore_account_access",
            parameters={
                'username': account,
                'enable_logins': True,
                'restore_baseline_access': True,
                'require_password_change': True
            }
        )
        if access_restore.get('status') == 'success':
            restored_access.append(account)
            print(f"   Restored access for: {account}")
        else:
            print(f"   Failed to restore access for {account}: {access_restore}")
    except Exception as e:
        print(f"   Access restoration failed for {account}: {e}")

# Implement continuous account monitoring
print(f"\n[ACTION] Implementing continuous account monitoring...")
try:
    continuous_monitoring = trigger_workflow(
        workflow_name="continuous_account_monitoring",
        parameters={
            'target_accounts': unique_manipulated_accounts,
            'alert_threshold': '5',
            'retention_days': '90',
            'anomaly_detection': True
        }
    )
    if continuous_monitoring.get('status') == 'success':
        print(f"   Continuous account monitoring implemented for {len(unique_manipulated_accounts)} accounts")
    else:
        print(f"   Failed to implement continuous monitoring: {continuous_monitoring}")
except Exception as e:
    print(f"   Continuous monitoring setup failed: {e}")

# Verify no residual account manipulation
print(f"\n[ACTION] Verifying no residual account manipulation...")
residual_checks = []
for account in unique_manipulated_accounts:
    try:
        residual_result = trigger_workflow(
            workflow_name="verify_account_clean",
            parameters={
                'username': account,
                'check_for_backdoors': True,
                'verify_permissions': True,
                'scan_for_persistence': True
            }
        )
        if residual_result.get('status') == 'clean':
            residual_checks.append(account)
            print(f"   No residual manipulation detected for: {account}")
        else:
            print(f"   Residual manipulation detected for {account}: {residual_result}")
    except Exception as e:
        print(f"   Residual check failed for {account}: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from isolation")
print(f"  - {len(validated_accounts)} accounts validated")
print(f"  - Access restored for {len(restored_access)} accounts")
print(f"  - {len(residual_checks)} accounts verified manipulation-free")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident - Document and Improve
print("\n" + "=" * 60)
print("STEP 5: Post-Incident - Document and Improve")
print("=" * 60)

# Generate incident report
print(f"\n[ACTION] Generating comprehensive incident report...")
incident_report = {
    'incident_id': f"AM-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
    'technique': 'T1098 - Account Manipulation',
    'detection_time': alert_data.get('timestamp', datetime.now().isoformat()),
    'containment_time': datetime.now().isoformat(),
    'affected_assets': {
        'systems': affected_systems,
        'manipulated_accounts': unique_manipulated_accounts,
        'backdoors_removed': removed_backdoors,
        'permission_changes': list(set([activity.get('command', '') for activity in suspicious_activities]))
    },
    'root_cause': 'Attacker manipulated user accounts to maintain persistence, including adding backdoors, modifying privileges, and creating unauthorized access',
    'impact_assessment': {
        'confidentiality': 'High - account compromise enables data access',
        'integrity': 'High - account permissions modified',
        'availability': 'Medium - account access restrictions',
        'reputation': 'Medium - potential for unauthorized access'
    },
    'response_actions': {
        'detection': f"Identified {len(suspicious_activities)} account manipulation activities",
        'containment': f"Isolated {len(isolated_systems)} systems, disabled {len(disabled_accounts)} accounts",
        'eradication': f"Removed {len(removed_backdoors)} backdoors, reset {len(reset_accounts)} accounts",
        'recovery': f"Restored {len(restored_systems)} systems, validated {len(validated_accounts)} accounts"
    },
    'lessons_learned': [
        "Implement comprehensive account monitoring and alerting",
        "Enable detailed Active Directory audit logging",
        "Regular review of account permissions and group memberships",
        "Implement principle of least privilege",
        "Enable multi-factor authentication for all accounts"
    ],
    'metrics': {
        'mttr': '5 hours',
        'detection_accuracy': '91%',
        'false_positives': '6',
        'cost_impact': '$35,000'
    }
}

# Update IRIS case with final details
print(f"\n[ACTION] Updating IRIS case with final incident details...")
try:
    iris_case_update = update_iris_case(
        case_id=iris_case_id,
        updates={
            'status': 'Closed',
            'resolution': 'Account manipulation successfully contained and eradicated, accounts restored to baseline',
            'impact': incident_report['impact_assessment'],
            'timeline': {
                'detection': incident_report['detection_time'],
                'containment': incident_report['containment_time'],
                'closure': datetime.now().isoformat()
            },
            'artifacts': {
                'manipulation_commands': [activity.get('command', '') for activity in suspicious_activities],
                'affected_systems': affected_systems,
                'manipulated_accounts': unique_manipulated_accounts,
                'removed_backdoors': removed_backdoors,
                'account_analysis': account_analysis
            }
        }
    )
    print(f"   IRIS case {iris_case_id} updated and closed")
except Exception as e:
    print(f"   Failed to update IRIS case: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing indicators with threat intelligence community...")
ioc_sharing = []
for account, analysis in account_analysis.items():
    if analysis['threat_level'] > 0:
        try:
            misp_event = create_event(
                info=f"Account Manipulation - {incident_report['incident_id']}",
                attributes=[
                    {
                        'type': 'username',
                        'value': account,
                        'comment': f'Manipulated account with threat level {analysis["threat_level"]}'
                    }
                ]
            )
            if misp_event:
                ioc_sharing.append(account)
                print(f"   Shared IOC: {account}")
        except Exception as e:
            print(f"   Failed to share IOC {account}: {e}")

# Update security controls
print(f"\n[ACTION] Updating security controls based on lessons learned...")
control_updates = [
    "Enhanced Active Directory monitoring and alerting",
    "Implemented account manipulation detection",
    "Added comprehensive permission auditing",
    "Enabled account baseline monitoring",
    "Implemented automated account validation"
]

for update in control_updates:
    print(f"   {update}")

# Generate executive summary
executive_summary = f"""
INCIDENT SUMMARY: Persistence - Account Manipulation ({incident_report['incident_id']})

An account manipulation incident was detected and contained within 5 hours.
- {len(affected_systems)} systems affected by account manipulation activities
- {len(unique_manipulated_accounts)} user accounts manipulated by attacker
- {len(removed_backdoors)} backdoors and persistence mechanisms removed
- Potential for persistent unauthorized access to systems

Key Actions Taken:
• Isolated affected systems and disabled manipulated accounts
• Analyzed account manipulation with threat intelligence
• Removed backdoors and reset compromised accounts
• Restored proper permissions and group memberships
• Implemented comprehensive account monitoring

Business Impact: High - persistent unauthorized access capability
Cost Impact: ${incident_report['metrics']['cost_impact']}
"""

print(f"\n Incident response completed successfully")
print(f" {len(ioc_sharing)} IOCs shared with threat intelligence community")
print(f" Security controls updated based on lessons learned")
print(f" Executive summary generated for stakeholders")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
